In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from pyspark.sql.types import *
from dataclasses import dataclass

In [0]:
%sql
CREATE TABLE IF NOT EXISTS proj.aviation.table_config (
    pipeline_name STRING,
    file_path STRING,
    header STRING,
    delimiter STRING,
    table_name STRING,
    schema_details MAP<STRING, STRING>,
    keys ARRAY<STRING>,
    partition_cols ARRAY<STRING>,
    mode STRING
)

In [0]:

config_schema = StructType(
    [
        StructField("pipeline_name", StringType(), False),
        StructField("file_path", StringType(), True),
        StructField("header", StringType(), True),
        StructField("delimiter", StringType(), True),
        StructField("table_name", StringType(), True),
        StructField(
            "schema_details", MapType(StringType(), StringType()), True
        ),
        StructField("keys", ArrayType(StringType()), True),
        StructField("partition_cols", ArrayType(StringType()), True),
        StructField("mode", StringType(), True),
    ]
)

flight_config = [
    {
        "pipeline_name": "flight_pipeline",
        "file_path": "/Volumes/proj/aviation/volume",
        "header": "true",
        "delimiter": ",",
        "table_name": "proj.aviation.flights",
        "schema_details": {
            "Year": "int",
            "Quarter": "int",
            "Month": "int",
            "DayofMonth": "int",
            "DayOfWeek": "int",
            "FlightDate": "date",
            "Reporting_Airline": "string",
            "Flight_Number_Reporting_Airline": "int",
            "Origin": "string",
            "OriginCityName": "string",
            "OriginState": "string",
            "Dest": "string",
            "DestCityName": "string",
            "DestState": "string",
            "CRSDepTime": "int",
            "DepTime": "float",
            "DepDelay": "float",
            "DepDelayMinutes": "float",
            "DepDel15": "int",
            "TaxiOut": "float",
            "WheelsOff": "float",
            "WheelsOn": "float",
            "TaxiIn": "float",
            "CRSArrTime": "int",
            "ArrTime": "float",
            "ArrDelay": "float",
            "ArrDelayMinutes": "float",
            "ArrDel15": "int",
            "Cancelled": "int",
            "CancellationCode": "string",
            "Diverted": "int",
            "CRSElapsedTime": "float",
            "ActualElapsedTime": "float",
            "AirTime": "float",
            "Distance": "float",
            "CarrierDelay": "float",
            "WeatherDelay": "float",
            "NASDelay": "float",
            "SecurityDelay": "float",
            "LateAircraftDelay": "float",
        },
        "keys": [
            "FlightDate",
            "Reporting_Airline",
            "Flight_Number_Reporting_Airline",
            "Origin",
            "Dest",
            "CRSDepTime",
        ],
        "partition_cols": ["Year", "Month"],
        "mode": "append",
    }
]


config_df = spark.createDataFrame(flight_config, config_schema)

target_table = DeltaTable.forName(spark, "proj.aviation.table_config")
(
    target_table.alias("tgt")
    .merge(
        config_df.alias("src"), "tgt.pipeline_name = src.pipeline_name"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
config_df.display()

pipeline_name,file_path,header,delimiter,table_name,schema_details,keys,partition_cols,mode
flight_pipeline,/Volumes/proj/aviation/volume,true,",",proj.aviation.flights,"Map(Flight_Number_Reporting_Airline -> int, TaxiOut -> float, DestCityName -> string, AirTime -> float, WeatherDelay -> float, CRSDepTime -> int, ArrDel15 -> int, Reporting_Airline -> string, FlightDate -> date, DestState -> string, Month -> int, Dest -> string, CarrierDelay -> float, OriginCityName -> string, DepDelayMinutes -> float, CRSElapsedTime -> float, Diverted -> int, NASDelay -> float, Origin -> string, DepDel15 -> int, Quarter -> int, DepTime -> float, LateAircraftDelay -> float, DayOfWeek -> int, Year -> int, OriginState -> string, SecurityDelay -> float, CancellationCode -> string, DayofMonth -> int, CRSArrTime -> int, Cancelled -> int, ArrDelayMinutes -> float, ArrDelay -> float, WheelsOn -> float, ArrTime -> float, ActualElapsedTime -> float, TaxiIn -> float, WheelsOff -> float, Distance -> float, DepDelay -> float)","List(FlightDate, Reporting_Airline, Flight_Number_Reporting_Airline, Origin, Dest, CRSDepTime)","List(Year, Month)",append
